In [ ]:
import pandas as pd
import load_TU_data
from otp_client import get_all_routes_for_mode, load_all_candidates
from find_similar_trip import find_similar_trip
from otp_utils import has_invalid_route_name, resolve_route_short_names


In [ ]:
tu_session, tu_tur, tu_deltur = load_TU_data.load_tu(
    data_dir = "~/O/TU_Rejseplan/Data/TU/",
    session_file = "tu_session_secret_2015_2025.xlsx",
    tur_file = "tu_tur_secret_2015_2025.xlsx",
    deltur_file = "tu_deltur_2015_2025.xlsx")

In [ ]:
tu_tur[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] >1 ) & (tu_tur["PtNumBoardings"]>1)]
tu_tur.loc[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] >1 ) & (tu_tur["PrimMode"]==32), ["SessionId","TurId"]] #35 (Telebus, Flextrafik) not included.

In [ ]:
i_TurId = 2648141
tu_deltur[tu_deltur["TurId"] == i_TurId]

In [ ]:
tu_tur[tu_tur["TurId"] == i_TurId]

In [ ]:
tu_deltur.loc[(tu_deltur["TurId"] == i_TurId) & (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])), "Route"]
      #35 (Telebus, Flextrafik) not included.

In [ ]:
tu_deltur.loc[(tu_deltur["TurId"]==i_TurId), "StageMode"]
#Need to split this up into direct (agress/egress) and transit.
#Remove duplicates
#Then map to GraphQL commands

In [ ]:
tu_tur.loc[tu_tur["TurId"] == i_TurId,["orig_lat", "orig_lon", "tiladrlat", "tiladrlon"]]

In [ ]:
#Transit mode mapping from TU to OTP
mode_map = {
#   TU: OTP
    31: "BUS",
    32: "S_TRAIN", #"S_TRAIN" added to OTP by Simun. Originally classified as RAIL
    33: "RAIL",
    34: "SUBWAY",
    37: "TRAM",
    41: "FERRY",
    35: "BUS" #Maybe remove trips with 35 #35: Telebus, Flextrafik - Behovsstyrede kollektive trafik #Often basically a taxi
}

In [ ]:
modes_list = (
    tu_deltur.loc[
        (tu_deltur["TurId"]==i_TurId) &
        (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])),
        "StageMode"
    ]
    .drop_duplicates()
    .map(mode_map)
    .tolist()
)
modes_json = [{"mode": m} for m in modes_list]
modes_json

In [ ]:
tu_deltur["Route"] = tu_deltur["Route"].astype(str)
route_short_name = tu_deltur.loc[
    (tu_deltur["TurId"]==i_TurId) &
    (tu_deltur["StageMode"].isin([31, 32, 33, 34, 37, 41])),
    "Route"
].drop_duplicates().astype(str).tolist()
print(route_short_name)

In [ ]:
tu_tur.loc[tu_tur["TurId"]==i_TurId, "depart_dt"]

In [ ]:
import otp_client
importlib.reload(otp_client)
from otp_client import get_all_routes_for_mode
# RAIL, TRAM and SUBWAY are missing route name in TU.
# So taking all routes for these modes. Which will be used when modes
# that do include route name in TU only can access those routes, but for
# those that do not, all routes will be used.
otp_mode_routes_cache = {}
otp_mode_routes_cache["RAIL"] = get_all_routes_for_mode("RAIL")
otp_mode_routes_cache["TRAM"] = get_all_routes_for_mode("TRAM")
otp_mode_routes_cache["SUBWAY"] = get_all_routes_for_mode("SUBWAY")

In [ ]:
otp_mode_routes_cache

In [ ]:
import otp_client
importlib.reload(otp_client)
from otp_client import graphql_json_request
import otp_parser
importlib.reload(otp_parser)
from otp_parser import json_to_df

tu_row = tu_tur.loc[tu_tur["TurId"] == i_TurId].iloc[0]

search_window = "PT10M"
resp_depart_dt = tu_tur.loc[tu_tur["TurId"]==i_TurId, "depart_dt"].iloc[0]
#Inital request
response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                direct=["WALK"], first=50, direct_only=False, transit_only=True,
                                search_window=search_window, url="http://localhost:8080/otp/gtfs/v1")
otp_candidates_df = json_to_df(response)
response_data = response.json()


In [ ]:
n_forward = len(response_data["data"]["planConnection"]["edges"])
hasNextPage = response_data["data"]["planConnection"]["pageInfo"]["hasNextPage"]
while n_forward < 50 and hasNextPage:
    # Check if all trips are within the search window
    otp_candidates_df["start_dt"] = pd.to_datetime(otp_candidates_df["start"]).dt.tz_convert('Europe/Copenhagen')
    trips_within_window = ((otp_candidates_df["start_dt"] - resp_depart_dt) <= pd.Timedelta(search_window)).all()
    if not trips_within_window:
        break

    endCursor = response_data["data"]["planConnection"]["pageInfo"]["endCursor"]
    response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                    direct=["WALK"], after=endCursor, first=50 - n_forward, direct_only=False,
                                    transit_only=True, search_window=search_window,
                                    url="http://localhost:8080/otp/gtfs/v1")
    otp_trip_candidates_forward = json_to_df(response)
    #Adjust iteration_id before concatenating
    offset = otp_candidates_df['iteration_id'].max() + 1
    otp_trip_candidates_forward['iteration_id'] = otp_trip_candidates_forward['iteration_id'] + offset
    otp_candidates_df = pd.concat([otp_candidates_df, otp_trip_candidates_forward]).reset_index(drop=True)

    response_data = response.json()
    hasNextPage = response_data["data"]["planConnection"]["pageInfo"]["hasNextPage"]
    n_forward += len(response_data["data"]["planConnection"]["edges"])


n_backward = 0
hasPreviousPage = response_data["data"]["planConnection"]["pageInfo"]["hasPreviousPage"]
while n_backward < 50 and hasPreviousPage:
        # Check if all trips are within the search window
    otp_candidates_df["start_dt"] = pd.to_datetime(otp_candidates_df["start"]).dt.tz_convert('Europe/Copenhagen')
    trips_within_window = ((otp_candidates_df["start_dt"] - resp_depart_dt) >= -pd.Timedelta(search_window)).all()
    if not trips_within_window:
        break

    startCursor = response_data["data"]["planConnection"]["pageInfo"]["startCursor"]
    response = graphql_json_request(tu_row=tu_row, modes_json=modes_json, route_short_name_json=route_short_name,
                                    direct=["WALK"], before=startCursor, last=50 - n_backward, direct_only=False,
                                    transit_only=True, search_window=search_window,
                                    url="http://localhost:8080/otp/gtfs/v1")
    otp_trip_candidates_backward = json_to_df(response)
    #Adjust iteration_id before concatenating
    offset = otp_candidates_df['iteration_id'].min() - otp_trip_candidates_backward['iteration_id'].max() - 1
    otp_trip_candidates_backward['iteration_id'] = otp_trip_candidates_backward['iteration_id'] + offset
    otp_candidates_df = pd.concat([otp_trip_candidates_backward, otp_candidates_df]).reset_index(drop=True)

    response_data = response.json()
    hasPreviousPage = response_data["data"]["planConnection"]["pageInfo"]["hasPreviousPage"]
    n_backward += len(response_data["data"]["planConnection"]["edges"])


In [ ]:
otp_candidates_df

In [ ]:
required_routes = set(route_short_name)
type(required_routes)

In [ ]:
iteration_ids_with_all_routes = (
    otp_candidates_df.groupby("iteration_id")["route_short_name"]
    .apply(lambda routes: required_routes.issubset(set(routes.astype(str))))
)
iteration_ids_with_all_routes

In [ ]:
otp_candidates_df = otp_candidates_df[
    otp_candidates_df["iteration_id"].isin(
        iteration_ids_with_all_routes[iteration_ids_with_all_routes].index
    )
].reset_index(drop=True)
otp_candidates_df

In [ ]:
#Calcuate the total deviation OTP trip and TU tur and find best matching trip
import find_similar_trip
importlib.reload(find_similar_trip)
from find_similar_trip import find_similar_trip
best_trip_candidate = find_similar_trip(tu_row, otp_candidates_df, arrival_dev_weight = 1)

In [ ]:
best_trip_candidate["TurId"] = i_TurId
best_trip_candidate

In [ ]:
import folium
import os

def plot_iteration(otp_trip_candidates, iteration_id, file_path=None, zoom_start=12):
    mode_colors = {
        "WALK": "red",
        "BUS": "blue",
        "RAIL": "green",
        "S_TRAIN": "green",
        "SUBWAY": "yellow",
    }

    trip_legs = (
        otp_trip_candidates.loc[otp_trip_candidates["iteration_id"] == iteration_id]
        .sort_values("leg_id")
        .copy()
    )

    if trip_legs.empty:
        raise ValueError(f"No legs found for iteration_id={iteration_id}")

    first_geometry = trip_legs["leg_geometry_length"].dropna().iloc[0]
    map_center = first_geometry[0]

    m = folium.Map(
        location=map_center,
        zoom_start=zoom_start,
        tiles="OpenStreetMap"
    )

    for _, leg in trip_legs.iterrows():
        mode = leg["mode"]
        geometry = leg["leg_geometry_length"]
        color = mode_colors.get(mode, "gray")

        folium.PolyLine(
            geometry,
            color=color,
            weight=5,
            opacity=0.8,
            tooltip=f"{mode} | leg_id={leg['leg_id']} | route={leg.get('route')}"
        ).add_to(m)

    if file_path is not None:
        file_path = os.path.expanduser(file_path)
        m.save(file_path)

    return m

In [ ]:
plot_iteration(otp_candidates_df, iteration_id=44)